In [20]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/validation.csv
/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/train.csv


# Introduction

This notebook contains code for a micro-Small Language Model (< 2 Million parameters) consisting of a Language Model with 1-2 transformer blocks and multi-headed, causal self-attention. This model is pretrained on the TinyStories dataset for the next-token prediction task. The pretrained model is then repeatedly input generated text to generate a short story.

# Steps:
1. **Data Exploration**
2. **Data Preprocessing**
     1. Tokenization
     2. Saving data to memory
     3. Input-ouput pairing
     4. Projection to Token Embedding Space
     5. Projection to Positional Embedding Space (for later; unclear on this)
3. **Building Model Architecture**
    1. Attention
    2. Feed-forward Network/ Multi-layer Perceptrons
    3. Layer Normalization
    4. Transformer
    5. Small Language Model (contains embedding projection, transformer blocks, LM Head, etc.)
4. **Pre-training**
    1. Defining cross-entropy loss function
    2. Updating parameters using backpropagation
    3. Pre-training using training data
    4. Validate model on validation dataset
    5. Visualize and analyze training and validation losses
    6. Hyperparameter tuning and repeat training and validation
5. **Short Story Generation**
6. **Explore and Evaluate Generated Output**


# Step 1: Data Exploration

In [21]:
if not os.path.exists('./data'):
    os.makedirs('./data')
    
kagglehub.dataset_download('thedevastator/tinystories-narrative-classification',
                           path = 'train.csv', output_dir = './data')
kagglehub.dataset_download('thedevastator/tinystories-narrative-classification',
                           path = 'validation.csv', output_dir = './data')

'/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/validation.csv'

In [22]:
train = pd.read_csv('/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/train.csv')
val = pd.read_csv('/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/validation.csv')

In [23]:
pd.options.display.max_rows = 500

print("Summary of training dataset:")
print(train.describe)

Summary of training dataset:
<bound method NDFrame.describe of                                                       text
0        One day, a little girl named Lily found a need...
1        Once upon a time, there was a little car named...
2        One day, a little fish named Fin was swimming ...
3        Once upon a time, in a land full of trees, the...
4        Once upon a time, there was a little girl name...
...                                                    ...
2119714  Once upon a time, in a small town, there lived...
2119715  Once upon a time, there was a little boy named...
2119716  Once upon a time, there was a big tree. Under ...
2119717  Once upon a time, there was a little girl name...
2119718  Once upon a time, there was an adorable little...

[2119719 rows x 1 columns]>


In [24]:
print("Summary of validation dataset:")
print(val.describe)

Summary of validation dataset:
<bound method NDFrame.describe of                                                     text
0      Spot. Spot saw the shiny car and said, "Wow, K...
1      Once upon a time, in a big forest, there lived...
2      Once upon a time, in a small yard, there was a...
3      Once upon a time, there was a thoughtful girl ...
4      Once upon a time, there was a kind farmer. He ...
...                                                  ...
21985  John was out in the backyard playing. He saw a...
21986  Once upon a time, there was a green vehicle. E...
21987  Once upon a time, there was a package. It was ...
21988  Once upon a time, there was a thin rhinoceros....
21989  Once there was a little girl named Sarah. She ...

[21990 rows x 1 columns]>


In [25]:
print('First story in training set:')
print(train.iloc[0]['text'])

First story in training set:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


## Select sample of 10 stories for initial run 

In [26]:
print('2 more stories from training set and validation set each:')
train_sample = train.sample(10, random_state = 1)
val_sample = val.sample(2, random_state = 1)

print(train_sample.iloc[0]['text'])
print('-------------X-------------')
print(train_sample.iloc[1]['text'])
print('-------------X-------------')
print(val_sample.iloc[0]['text'])
print('-------------X-------------')
print(val_sample.iloc[1]['text'])

2 more stories from training set and validation set each:
Once upon a time, there was a big elephant named Ellie. Ellie loved to walk in the forest and eat leaves. One day, Ellie met a little bird named Billy. Billy said, "Hi, Ellie! You are so big and graceful. Can I walk with you?"

Ellie was happy to have a friend to walk with. They walked and walked until they saw a big snake. The snake was not happy and said, "Get out of my way! I am going to poke you with my sharp teeth!"

Ellie and Billy were scared and did not know what to do. Ellie said, "Please don't poke us. We just want to walk." The snake thought for a moment and said, "Okay, you can walk. But be careful, there are many dangers in the forest."

Ellie and Billy were happy to continue their walk and thanked the snake for being kind. They walked until they reached a river. Ellie said, "Let's poke our trunks in the water and have a drink." They drank the water and continued their walk, happy to have each other's company.
-----

## Tokenize data 

In [27]:
!pip install git+https://github.com/huggingface/transformers

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-2nxt2bru
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-2nxt2bru
  Resolved https://github.com/huggingface/transformers to commit 22cfdfaad99038416b56d21e35b0d32d4f4ee2d3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
^C
ERROR: Operation cancelled by user


In [28]:

from transformers import GPT2Tokenizer, BatchEncoding

tokenizer = GPT2Tokenizer.from_pretrained('openai-community/gpt2', add_eos_token = True)
print('Vocabulary size = ', tokenizer.vocab_size)

trainTokens = tokenizer(train_sample['text'].tolist())
valTokens = tokenizer(val_sample['text'].tolist())

print(trainTokens[0])
print('---------------X--------------')
print(valTokens[0])
print('---------------X--------------')
print(trainTokens['input_ids'][0])
print('---------------X--------------')
print(len(trainTokens['input_ids']))
print('---------------X--------------')
print(tokenizer.decode(trainTokens['input_ids'][0]))

Vocabulary size =  50257
Encoding(num_tokens=231, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])
---------------X--------------
Encoding(num_tokens=134, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])
---------------X--------------
[7454, 2402, 257, 640, 11, 612, 373, 257, 1263, 20950, 3706, 44801, 13, 44801, 6151, 284, 2513, 287, 262, 8222, 290, 4483, 5667, 13, 1881, 1110, 11, 44801, 1138, 257, 1310, 6512, 3706, 15890, 13, 15890, 531, 11, 366, 17250, 11, 44801, 0, 921, 389, 523, 1263, 290, 44363, 13, 1680, 314, 2513, 351, 345, 1701, 198, 198, 30639, 494, 373, 3772, 284, 423, 257, 1545, 284, 2513, 351, 13, 1119, 6807, 290, 6807, 1566, 484, 2497, 257, 1263, 17522, 13, 383, 17522, 373, 407, 3772, 290, 531, 11, 366, 3855, 503, 286, 616, 835, 0, 314, 716, 1016, 284, 22620, 345, 351, 616, 7786, 9941, 2474, 198, 198, 30639, 494, 290, 15890, 547, 12008, 290, 750, 407, 760, 644, 284, 466, 13, 44801, 

## Create input-output pairs 

For the next token prediction task, a **sliding window** is defined, such that given the tokens in the window, i.e. the **context**, the next token in the sequence is considered to be the expected output. The max number of tokens that can be in the window is the **context size**. Tokenized data is grouped into these input (the tokens in the window) and output (the next expected token) pairs. These groups are called **chunks**.

In [38]:
def io_pairing(data):
    inputTokens, outputTokens = [], []
    # Leave out the last token;it is never input to the model
    for document in data:    
        inputTokens.extend(document[0:len(document) - 1])
        outputTokens.extend(document[1:len(document)])

    return inputTokens, outputTokens

In [44]:
inputTokens, outputTokens = io_pairing(trainTokens['input_ids'])
print(inputTokens[0:4])
print(outputTokens[0:4])
print(inputTokens[len(inputTokens) - 4:len(inputTokens)])
print(outputTokens[len(outputTokens) - 4:len(outputTokens)])

[7454, 2402, 257, 640]
[2402, 257, 640, 11]
[923, 607, 1110, 13]
[607, 1110, 13, 50256]


In [45]:
inputValTokens, outputValTokens = io_pairing(valTokens['input_ids'])

### Batch Data to Prepare it for the Embedding Layer of the Model

In [46]:
import torch.nn.functional as F
import torch

In [ ]:
class Dataset:
    def __init__(self, inputData, targetData, batchSize, contextSize):
        assert inputData.shape[0] // batchSize
        numOfBatches = inputData.shape[0] / batchSize
        
        inputData = torch.tensor(inputData)
        outputData = torch.tensor(outputData)
        self.inputBatches = []
        self.targetBatches = []

        inputBatch = torch.zeros((batchSize, contextSize))
        targetBatch = torch.zeros(batchSize)
        for i in range(numOfBatches):
            startOfBatch = i * batchSize
            for j in range(batchSize):
                inputBatch[j, :] = inputData[startOfBatch + j:contextSize]
                targetBatch[j] = outputData[startOfBatch + j]
            self.inputBatches.append(inputBatch)
            self.targetBatches.append(targetBatch)

    def __len__(self):
        return len(self.inputBatches)

    def __get_item__(self, index):
        return self.inputBatches[index], self.targetBatches[index]

In [ ]:
def get_batches(inputData, outputData, batchSize, contextSize):
    inputData = torch.tensor(inputData)
    outputData = torch.tensor(outputData)
    batch = [torch.zeros((batchSize, contextSize)), torch.zeros(batchSize)]
    for 

## Project to Embedding Space

A set of numbers, such as these **tokens**, carry no information whatsoever to a computer. To imbue these tokens with some **semantic meaning**, we project them to a higher order **embedding space**. Thus, each token becomes a vector of *n* dimensions, where *n* is the dimension of the embedding space.

These weights, first initialized randomly, will be learned by the Language Model during training.

In [34]:
def generate_embd(vocabSize, embedDim, inputTokens, outputTokens):
    embedMatrix = torch.rand(vocabSize, embedDim) # save as model parameter
    embedInput = torch.matmul(F.one_hot(inputTokens, vocabSize), embedMatrix)
    embedOutput = F.one_hot(outputTokens, vocabSize) * embedMatrix

    return embedMatrix, embedInput, embedOutput

In [35]:
embedMatrix, embedInput, embedOutput = generate_embd(tokenizer.vocab_size, 
                                                     128, 
                                                     torch.from_numpy(inputTokens), 
                                                     torch.from_numpy(outputTokens))
embedMatrix.shape
embedInput.shape
embedOutput.shape

RuntimeError: expected m1 and m2 to have the same dtype, but got: long int != float